# Vroegsignalering — Effectiveness Analysis

## Purpose
Early-warning systems for problematic debt are designed to reach households before their situation
escalates. But reaching someone is only the first step — the real question is whether that contact actually
**helps people get back on track**. This notebook looks beyond *"did we make contact?"* to ask
*"did it work?"*, by measuring whether households stay out of trouble after a case is closed.

## What we measure
We define success as a household **not returning with a new signal** in the months after its case is closed.
Around that single idea, the notebook builds up from a broad picture to a focused analysis:

- **The bigger picture** — how many signals come in, who reports them, and how cases are handled and resolved.
- **What actually works** — how the way a household is contacted (the method, the timing, their response)
  relates to whether they return, using statistical models with honest caveats about what the data can and
  cannot prove.

**Privacy first** - Although this notebook works with anonymised data, it never prints or exports individual records; every result is an aggregate statistic or chart.

## The data
Each export shares the same three core tabs — **Signalen** (signals), **Meldingen** (reports/cases) and
**Melders** (reporters) — alongside contact tabs that vary by municipality. The pieces fit together through
shared reference numbers, letting us follow a household's journey from first signal to final outcome.


## 0. Install Dependencies

Run this cell once to install all required Python libraries.
If the packages are already present in your environment this step will be skipped automatically.


In [ ]:
%pip install pandas numpy matplotlib seaborn openpyxl statsmodels scipy --quiet


## 1. Setup

Import all required libraries and configure chart styling.
**Update the `FILE` variable** on the second line of the cell below if the export file has a different name or location.


In [ ]:
# Force a NON-blocking, headless plotting backend. Without this, plt.show() can
# block forever waiting for a GUI window (TkAgg/QtAgg/MacOSX) — which makes the
# first plotting cell appear to "hang". 'inline' renders in the notebook and is
# non-blocking; if IPython is unavailable we fall back to the headless 'Agg'.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    import matplotlib
    matplotlib.use('Agg')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
FILE     = Path('bi_export_sample.xlsx')   # ← update to real file path if needed
DATE_FMT = '%d-%m-%Y'

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid')
COLORS = sns.color_palette('Set2')

# ══════════════════════════════════════════════════════════════════════════════
#  RESULTS-FILE CAPTURE
#  Every print(), every table shown with display(), and every chart shown with
#  plt.show() is automatically recorded AND written to a single self-contained
#  HTML file (elsa_results_report.html) at the end (Section 15).
#  Send that one file back — it opens in any browser and embeds all charts.
#  Cell outputs still appear in the notebook as normal.
# ══════════════════════════════════════════════════════════════════════════════
import io as _io, base64 as _b64, html as _html, datetime as _dt, builtins as _bi
from IPython.display import display as _ipy_display   # the REAL display (safe to re-import)

REPORT_PATH   = (FILE.parent if hasattr(FILE, 'parent') else Path('.')) / 'elsa_results_report.html'
_REPORT_BLOCKS = []

def _report_add(html_str):
    _REPORT_BLOCKS.append(html_str)

def _report_section(title):
    _report_add(f"<h2 class='sec'>{_html.escape(str(title))}</h2>")

# Tee print() → real stdout + report. builtins.print is always the genuine print,
# so re-running this cell never stacks wrappers on top of each other.
_orig_print = _bi.print
def print(*args, **kwargs):
    _orig_print(*args, **kwargs)
    sep = kwargs.get('sep', ' '); end = kwargs.get('end', '\n')
    _report_add(f"<pre>{_html.escape(sep.join(str(a) for a in args) + end)}</pre>")

# Tee display() → real display + report. _ipy_display is imported straight from
# IPython above, NOT taken from the namespace, so it can never become this wrapper
# (which previously caused infinite recursion when the cell was run twice).
def display(obj, *args, **kwargs):
    _ipy_display(obj, *args, **kwargs)
    try:
        if isinstance(obj, pd.DataFrame):
            _report_add(obj.to_html(border=0))
        elif isinstance(obj, pd.Series):
            _report_add(obj.to_frame().to_html(border=0))
        else:
            _report_add(f"<pre>{_html.escape(str(obj))}</pre>")
    except Exception:
        pass

# Tee plt.show() → embed the current figure as a PNG. Guarded so that re-running
# this cell does not wrap an already-wrapped show() (which would double-capture).
if not getattr(plt.show, '_elsa_wrapped', False):
    _orig_show = plt.show
    def _report_show(*args, **kwargs):
        fig = plt.gcf()
        try:
            buf = _io.BytesIO()
            fig.savefig(buf, format='png', dpi=110, bbox_inches='tight')
            buf.seek(0)
            _report_add(f"<img src='data:image/png;base64,{_b64.b64encode(buf.read()).decode()}'/>")
        except Exception:
            pass
        _orig_show(*args, **kwargs)
    _report_show._elsa_wrapped = True
    plt.show = _report_show


### 1a. Load Data

Read all tabs from the Excel file, parse date columns to a consistent format, and define a helper function that converts Dutch `Ja`/`Nee` text values to booleans.
Row counts are printed at the end to confirm the file loaded correctly — no actual data values are shown.


In [ ]:
_report_section('1a. Load Data')
xls = pd.ExcelFile(FILE)

signalen  = pd.read_excel(xls, 'Signalen')
meldingen = pd.read_excel(xls, 'Meldingen')
melders   = pd.read_excel(xls, 'Melders')

FIXED_TABS = {'Signalen', 'Meldingen', 'Melders'}
extra_tabs = {name: pd.read_excel(xls, name)
              for name in xls.sheet_names if name not in FIXED_TABS}

# ── Parse date columns (robust to already-datetime cells and other formats) ───
RAW_DATE_SAMPLES = {}   # original text of date columns, captured BEFORE parsing (for v2)
def parse_dates(df, cols):
    for c in cols:
        if c not in df.columns:
            continue
        s = df[c]
        if pd.api.types.is_datetime64_any_dtype(s):
            continue                      # Excel already gave real date cells
        ex = s.dropna().astype(str).head(3).tolist()   # example raw values reveal the format
        if ex:
            RAW_DATE_SAMPLES.setdefault(c, ex)
        parsed = pd.to_datetime(s, format=DATE_FMT, errors='coerce')
        if parsed.notna().mean() < 0.5:   # fixed format matched almost nothing
            parsed = pd.to_datetime(s, dayfirst=True, errors='coerce')
        df[c] = parsed

parse_dates(signalen,  ['Datum melding', 'Datum doorgezet', 'Startdatum contact'])
parse_dates(meldingen, ['Afgerond d.d.', 'Gezien d.d.'])
for df in extra_tabs.values():
    date_cols = [c for c in df.columns if 'datum' in c.lower() or c.lower() == 'datum']
    parse_dates(df, date_cols)

# ── Ja/Nee → boolean helper ───────────────────────────────────────────────────
def ja_nee(series):
    return series.astype(str).str.strip().str.lower().map({'ja': True, 'nee': False})

# ── Confirm tabs loaded (row counts only — no raw data) ───────────────────────
all_tabs = [('Signalen', signalen), ('Meldingen', meldingen), ('Melders', melders),
            *extra_tabs.items()]
for name, df in all_tabs:
    print(f"  {name:<35} {len(df):>7,} rows")


## 2. Dataset Profile — Knowledge Base for the Next Version

The real export is seen **only once** when this notebook runs, and only this report comes back.
So this section banks a thorough, privacy-safe **profile of the dataset** — schema, exact category labels,
value ranges, date coverage and join integrity — so the next version of the analysis can be written
**without needing to see the raw data again**.

Everything here is aggregate metadata: counts, percentages, ranges and (for low-cardinality, non-identifier
columns) the list of distinct category labels. **No individual records are shown.** A machine-readable JSON
copy of the whole profile is printed at the end of the section (2e) so it can be copied out and reused.


### 2a. Schema — tabs, columns, dtypes, missingness, cardinality

In [ ]:
_report_section('2a. Schema — tabs, columns, dtypes, missingness, cardinality')
# Columns that must never have their raw values shown (identifiers / personal / free text)
SENSITIVE_COLS = {
    'Hash', 'SignaalID', 'Meldingnummer', 'ID', 'Melder ID', 'Postcode',
    'Naam medewerker', 'Melder', 'Melder (submerk)', 'Onderwerp',
    'Toelichting bij de FollowUp', 'Verwijzing anders namelijk', 'Waarnaar is verwezen',
}
CATEGORICAL_MAX = 50   # only list distinct values for columns with at most this many

def schema_table(df):
    return pd.DataFrame({
        'dtype':    df.dtypes.astype(str),
        'non_null': df.count(),
        'null_%':   (df.isnull().sum() / max(len(df), 1) * 100).round(1),
        'n_unique': df.nunique(dropna=True),
    })

for name, df in all_tabs:
    print(f"\n══ {name}: {len(df):,} rows × {df.shape[1]} cols ══")
    display(schema_table(df))


### 2b. Exact Category Labels & Frequencies

For every **low-cardinality, non-identifier** column, the full set of distinct values and their counts.
This is the most valuable part for writing correct mappings/normalisers next time — it pins down the exact
spelling and casing of every contact method, outcome, Ja/Nee token, etc.


In [ ]:
_report_section('2b. Exact Category Labels & Frequencies')
for name, df in all_tabs:
    shown_any = False
    for col in df.columns:
        if col in SENSITIVE_COLS:
            continue
        nu = df[col].nunique(dropna=True)
        if nu < 1 or nu > 1000:
            continue
        if not shown_any:
            print(f"\n══ {name} ══")
            shown_any = True
        if nu <= CATEGORICAL_MAX:
            vc = df[col].value_counts(dropna=False)
            vc.index = vc.index.map(lambda x: '(blank)' if pd.isna(x) else str(x))
            print(f"\n• {col}  ({nu} distinct):")
            print(vc.to_string())
        else:
            # 50 < distinct <= 1000: list only the most common values
            vc = df[col].value_counts().head(15)
            print(f"\n• {col}  ({nu} distinct — showing top 15):")
            print(vc.to_string())


### 2c. Numeric Ranges & Date Coverage

Ranges for numeric columns (to design bands from reality) and coverage / parse-success for date columns.


In [ ]:
_report_section('2c. Numeric Ranges & Date Coverage')
for name, df in all_tabs:
    num = df.select_dtypes(include='number')
    num = num[[c for c in num.columns if c not in SENSITIVE_COLS]]
    if num.shape[1]:
        print(f"\n══ {name} — numeric columns ══")
        display(num.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(2))

print("\n══ Date columns — coverage & parse success ══")
date_cols = {'Signalen': ['Datum melding', 'Datum doorgezet', 'Startdatum contact'],
             'Meldingen': ['Afgerond d.d.', 'Gezien d.d.']}
tab_lookup = dict(all_tabs)
for tab, cols in date_cols.items():
    if tab not in tab_lookup:
        continue
    df = tab_lookup[tab]
    for c in cols:
        if c in df.columns and pd.api.types.is_datetime64_any_dtype(df[c]):
            s = df[c]
            ok = s.notna().mean() * 100
            print(f"  {tab}.{c:<20} parsed={ok:5.1f}%  "
                  f"range={s.min().date() if s.notna().any() else 'n/a'} → "
                  f"{s.max().date() if s.notna().any() else 'n/a'}")

print("\n══ Raw date formats (original text, before parsing) ══")
if RAW_DATE_SAMPLES:
    for _col, _ex in RAW_DATE_SAMPLES.items():
        print(f"  {_col:<22} e.g. {_ex}")
else:
    print("  All date columns were already proper date cells (no text parsing needed).")

### 2d. Join Integrity & Structural Distributions

How the tabs link to each other (orphan/duplicate rates) and the structural distributions that determine
what is analysable next time: signals per household, contact attempts per case, follow-up availability,
and cases per municipality.


In [ ]:
_report_section('2d. Join Integrity & Structural Distributions')
print("── Join integrity ──")
sig_with_mld = signalen['Meldingnummer'].notna().mean() * 100
print(f"  Signalen with a Meldingnummer (linkable to Meldingen): {sig_with_mld:.1f}%")

mld_keys = set(meldingen['Meldingnummer'].dropna())
for tab in ['Tussenresultaat', 'Eindresultaat', 'Follow-Up resultaat']:
    if tab in extra_tabs and 'Meldingnummer' in extra_tabs[tab].columns:
        ids = extra_tabs[tab]['Meldingnummer'].dropna()
        match = ids.isin(mld_keys).mean() * 100 if len(ids) else 0
        print(f"  {tab}: {len(ids):,} rows, {match:.1f}% match a Meldingen case")

if 'Melder ID' in signalen.columns:
    melder_ids = set(melders['ID'].dropna())
    m = signalen['Melder ID'].dropna()
    print(f"  Signalen.Melder ID matching Melders registry: "
          f"{m.isin(melder_ids).mean()*100:.1f}%")

dup_mld = int(meldingen['Meldingnummer'].duplicated().sum())
print(f"  Duplicate Meldingnummer in Meldingen: {dup_mld:,}")

print("\n── Recidive structure (signals per household Hash) ──")
sph = signalen.dropna(subset=['Hash']).groupby('Hash').size()
if len(sph):
    print(f"  Households (unique Hash): {len(sph):,}")
    print(f"  Signals per household: mean={sph.mean():.2f}  median={sph.median():.0f}  max={sph.max()}")
    multi = (sph > 1).mean() * 100
    print(f"  Households with >1 signal (recidive present): {multi:.1f}%")

print("\n── Contact attempts per case ──")
att = []
for tab in ['Tussenresultaat', 'Eindresultaat']:
    if tab in extra_tabs:
        att.append(extra_tabs[tab][['Meldingnummer']])
if att:
    apc = pd.concat(att).dropna().groupby('Meldingnummer').size()
    print(f"  Cases with ≥1 attempt: {len(apc):,}")
    print(f"  Attempts per case: mean={apc.mean():.2f}  median={apc.median():.0f}  max={apc.max()}")

print("\n── Follow-up availability for recidive windows ──")
if 'Afgerond d.d.' in meldingen.columns and pd.api.types.is_datetime64_any_dtype(meldingen['Afgerond d.d.']):
    dmax = signalen['Datum melding'].max()
    closed_ct = ja_nee(meldingen['Afgerond'])
    for label, days in {'6 months': 182, '9 months': 273}.items():
        ok = (closed_ct.fillna(False) &
              meldingen['Afgerond d.d.'].notna() &
              ((meldingen['Afgerond d.d.'] + pd.Timedelta(days=days)) <= dmax)).sum()
        print(f"  Closed cases with ≥{label} follow-up: {ok:,}")

print("\n── Cases per municipality (multilevel feasibility) ──")
if 'Gemeente' in meldingen.columns:
    cpm = meldingen['Gemeente'].value_counts()
    print(f"  Municipalities: {cpm.size}  |  cases/municipality "
          f"mean={cpm.mean():.0f} median={cpm.median():.0f} min={cpm.min()} max={cpm.max()}")

print("\n── One household per case? (Meldingnummer -> Hash) ──")
_sl  = signalen.dropna(subset=['Meldingnummer', 'Hash'])
_hpc = _sl.groupby('Meldingnummer')['Hash'].nunique()
_multi = int((_hpc > 1).sum())
print(f"  Cases with linked signals: {len(_hpc):,}")
print(f"  ...mapping to >1 distinct household Hash: {_multi:,} "
      f"({_multi / max(len(_hpc), 1) * 100:.2f}%)")
print("  OK - one household per case; case-level recidive logic holds." if _multi == 0
      else "  WARNING - some cases span multiple households; revisit the one-Hash-per-case assumption.")

print("\n── Signalen -> Meldingen referential integrity ──")
_sig_mld = signalen['Meldingnummer'].dropna()
_orphan  = int((~_sig_mld.isin(mld_keys)).sum())
print(f"  Signals carrying a Meldingnummer: {len(_sig_mld):,}")
print(f"  ...whose Meldingnummer is NOT found in Meldingen (orphans): {_orphan:,} "
      f"({_orphan / max(len(_sig_mld), 1) * 100:.2f}%)")

## 3. Signal Volume & Distribution

How many signals arrive per month, and which signal types are most common?


In [ ]:
_report_section('3. Signal Volume & Distribution')
# Use only valid dates; convert to monthly Timestamps for a clean, fast x-axis.
# (Garbage/NaT dates are dropped so a few bad rows can't blow up the plot range.)
_valid_dates = signalen['Datum melding'].dropna()
monthly = (_valid_dates.dt.to_period('M').dt.to_timestamp()
           .value_counts().sort_index().rename('signals'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if len(monthly) > 0:
    monthly.plot(ax=axes[0], marker='o', color=COLORS[0], linewidth=2)
axes[0].set_title('Signal Volume per Month')
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of signals')

type_counts = signalen['Type melding'].value_counts()
type_counts.plot.barh(ax=axes[1], color=COLORS[1])
axes[1].set_title('Signal Type (Type melding)')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

print(f"Total signals      : {len(signalen):,}")
print(f"Valid signal dates : {len(_valid_dates):,} "
      f"({len(_valid_dates)/max(len(signalen),1)*100:.1f}%)")
if len(_valid_dates) > 0:
    print(f"Date range         : {_valid_dates.min().date()} → {_valid_dates.max().date()}")
else:
    print("Date range         : no parseable dates in 'Datum melding'")
print()
print("Signal type counts:")
print(type_counts.to_string())
print()
print("Signals per month (values behind the monthly volume chart):")
print(monthly.to_string() if len(monthly) else "(no parseable dates)")


### 3a. Reporter (Melder) Analysis

Which organisations are sending signals, and in what volume?

In [ ]:
_report_section('3a. Reporter (Melder) Analysis')
sig_with_melder = signalen.merge(
    melders.rename(columns={'ID': 'Melder ID'}), on='Melder ID', how='left'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_reporters = sig_with_melder['Melder'].value_counts().head(15)
top_reporters.sort_values().plot.barh(ax=axes[0], color=COLORS[2])
axes[0].set_title('Top 15 Reporters by Signal Volume')
axes[0].set_xlabel('Number of signals')

type_melder = sig_with_melder['Type melder'].value_counts()
type_melder.plot.pie(ax=axes[1], autopct='%1.1f%%',
                     colors=sns.color_palette('Set2', len(type_melder)),
                     startangle=90)
axes[1].set_title('Reporter Type Distribution (Type melder)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("Top 15 reporters by signal volume (values behind the left chart):")
print(top_reporters.sort_values(ascending=False).rename('signals').to_string())
print()
print("Reporter type — signal counts:")
print(
    type_melder.rename('signals')
    .to_frame()
    .assign(pct_of_total=(type_melder / len(signalen) * 100).round(1))
    .to_string()
)


## 4. Signal → Report Conversion

A signal is "forwarded" (`Doorgezet = Ja`) when the municipality decides it warrants follow-up.
We measure the overall forwarding rate and whether it differs by signal type or reporter type.


In [ ]:
_report_section('4. Signal → Report Conversion')
signalen['doorgezet_bool'] = ja_nee(signalen['Doorgezet'])
signalen['has_report']     = signalen['Meldingnummer'].notna()

total       = len(signalen)
n_forwarded = int(signalen['doorgezet_bool'].sum())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Overall pie
doorgezet_counts = signalen['Doorgezet'].value_counts(dropna=False)
doorgezet_counts.index = doorgezet_counts.index.fillna('(blank)')
doorgezet_counts.plot.pie(ax=axes[0], autopct='%1.1f%%',
                          colors=[COLORS[1], COLORS[0], COLORS[3]], startangle=90)
axes[0].set_title('Forwarded to Report?\n(Doorgezet)')
axes[0].set_ylabel('')

# By signal type
type_conv = (signalen.groupby('Type melding')['doorgezet_bool']
             .agg(forwarded='sum', total='count'))
type_conv['rate_%'] = type_conv['forwarded'] / type_conv['total'] * 100
type_conv['rate_%'].sort_values().plot.barh(ax=axes[1], color=COLORS[3])
axes[1].axvline(n_forwarded / total * 100, color='red', linestyle='--',
                linewidth=1.5, label=f'Overall {n_forwarded/total*100:.1f}%')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title('Conversion Rate by Signal Type')
axes[1].set_xlabel('% Forwarded')
axes[1].legend(fontsize=9)

# By reporter type
if 'Type melder' in sig_with_melder.columns:
    sig_with_melder['doorgezet_bool'] = ja_nee(sig_with_melder['Doorgezet'])
    type_melder_conv = (sig_with_melder.groupby('Type melder')['doorgezet_bool']
                        .agg(forwarded='sum', total='count'))
    type_melder_conv['rate_%'] = (type_melder_conv['forwarded']
                                  / type_melder_conv['total'] * 100)
    type_melder_conv['rate_%'].sort_values().plot.barh(ax=axes[2], color=COLORS[4])
    axes[2].axvline(n_forwarded / total * 100, color='red', linestyle='--', linewidth=1.5)
    axes[2].xaxis.set_major_formatter(mtick.PercentFormatter())
    axes[2].set_title('Conversion Rate by Reporter Type')
    axes[2].set_xlabel('% Forwarded')

plt.tight_layout()
plt.show()

print(f"Overall forwarding rate: {n_forwarded:,} / {total:,} = {n_forwarded/total*100:.1f}%")
print()
print("Conversion by signal type:")
print(type_conv.sort_values('rate_%', ascending=False)
      .assign(**{'rate_%': type_conv['rate_%'].round(1)})
      .to_string())
print()
print("Forwarded vs not (Doorgezet) - counts behind the pie:")
print(doorgezet_counts.rename('count').to_string())
if 'Type melder' in sig_with_melder.columns:
    print()
    print("Conversion by reporter type (values behind the right chart):")
    print(type_melder_conv.sort_values('rate_%', ascending=False)
          .assign(**{'rate_%': type_melder_conv['rate_%'].round(1)}).to_string())


## 5. Contact Effectiveness

Once a report exists, the municipality tries to reach the client. We look at:
- Overall reach rate measured from **contact attempts** (`Persoon bereikt`).
  Note: `Bereikt` in the Meldingen tab is not used by this municipality (≈100% "N.v.t."),
  so reach is taken from the contact tabs instead.
- Reach rate per contact method and time of day
- How many attempts are typically needed


In [ ]:
_report_section('5. Contact Effectiveness')
# Reach is measured from contact attempts (Persoon bereikt); Meldingen 'Bereikt' is
# ~100% 'N.v.t.' for this municipality and is therefore not used.
contact_dfs = []
for tab_name in ['Tussenresultaat', 'Eindresultaat']:
    if tab_name in extra_tabs:
        df = extra_tabs[tab_name].copy()
        df['_tab'] = tab_name
        contact_dfs.append(df)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

if contact_dfs:
    contacts = pd.concat(contact_dfs, ignore_index=True)
    contacts['bereikt_bool'] = ja_nee(contacts['Persoon bereikt'])

    # Case-level reach: a case counts as reached if ANY attempt reached the person
    case_reach = contacts.groupby('Meldingnummer')['bereikt_bool'].any()
    meldingen['reached_case'] = meldingen['Meldingnummer'].map(case_reach).fillna(False)
    attempt_rate = contacts['bereikt_bool'].mean() * 100

    reached_ct = pd.Series({'Reached': int(case_reach.sum()),
                            'Not reached': int((~case_reach).sum())})
    reached_ct.plot.pie(ax=axes[0], autopct='%1.1f%%',
                        colors=[COLORS[0], COLORS[1]], startangle=90)
    axes[0].set_title('Client Reached? (case-level)\nany successful attempt'); axes[0].set_ylabel('')

    method_reach = (contacts.groupby('Wijze van contactpoging')['bereikt_bool']
                    .agg(reach_rate='mean', n_attempts='count'))
    method_reach['reach_rate'] *= 100
    method_reach['reach_rate'].sort_values().plot.barh(ax=axes[1], color=COLORS[2])
    axes[1].axvline(attempt_rate, color='red', linestyle='--', linewidth=1.5, label='Overall avg')
    axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
    axes[1].set_title('Reach Rate by Contact Method'); axes[1].set_xlabel('% Person Reached'); axes[1].legend(fontsize=9)

    dagdeel_reach = (contacts.groupby('Dagdeel')['bereikt_bool']
                     .agg(reach_rate='mean', n_attempts='count'))
    dagdeel_reach['reach_rate'] *= 100
    dagdeel_reach['reach_rate'].sort_values().plot.barh(ax=axes[2], color=COLORS[4])
    axes[2].axvline(attempt_rate, color='red', linestyle='--', linewidth=1.5)
    axes[2].xaxis.set_major_formatter(mtick.PercentFormatter())
    axes[2].set_title('Reach Rate by Time of Day (Dagdeel)'); axes[2].set_xlabel('% Person Reached')

    plt.tight_layout(); plt.show()

    print("Reach is measured from contact attempts (Persoon bereikt); Meldingen 'Bereikt' is unused (~100% 'N.v.t.').")
    print(f"Case-level reach rate    : {case_reach.mean()*100:.1f}%  "
          f"({int(case_reach.sum()):,} of {case_reach.size:,} cases with >=1 attempt)")
    print(f"Attempt-level reach rate : {attempt_rate:.1f}%  ({len(contacts):,} attempts)")
    print()
    print("Reach rate by contact method:")
    print(method_reach.sort_values('reach_rate', ascending=False)
          .assign(reach_rate=method_reach['reach_rate'].round(1)).to_string())
    print()
    print("Reach rate by time of day (Dagdeel) - values behind the right chart:")
    print(dagdeel_reach.sort_values('reach_rate', ascending=False)
          .assign(reach_rate=dagdeel_reach['reach_rate'].round(1)).to_string())
else:
    meldingen['reached_case'] = False
    for ax in axes:
        ax.text(0.5, 0.5, 'No contact-attempt tabs\nfound in this export',
                ha='center', va='center', transform=ax.transAxes, fontsize=11)
    plt.tight_layout(); plt.show()
    print("No contact-attempt tabs found in this export.")


## 6. Client Response & Outcomes

Once the client is reached, do they want help, and what is the final outcome?


In [ ]:
_report_section('6. Client Response & Outcomes')
if 'Eindresultaat' in extra_tabs:
    eind = extra_tabs['Eindresultaat'].copy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    hulp_vc = eind['Wil de klant hulp'].value_counts(dropna=False)
    hulp_vc.index = hulp_vc.index.fillna('(not recorded)')
    hulp_vc.plot.pie(ax=axes[0], autopct='%1.1f%%',
                     colors=sns.color_palette('Set2', len(hulp_vc)), startangle=90)
    axes[0].set_title('Does the Client Want Help?\n(Wil de klant hulp)')
    axes[0].set_ylabel('')

    result_vc = eind['Wat is het eindresultaat'].value_counts(dropna=False)
    result_vc.index = result_vc.index.fillna('(not recorded)')
    result_vc.sort_values().plot.barh(ax=axes[1], color=COLORS[0])
    axes[1].set_title('Final Outcome (Wat is het eindresultaat)')
    axes[1].set_xlabel('Count')

    plt.tight_layout()
    plt.show()

    print("Does the client want help? (Wil de klant hulp) - counts behind the pie:")
    print(hulp_vc.rename('count').to_frame()
          .assign(pct_=(hulp_vc / hulp_vc.sum() * 100).round(1)).to_string())
    print()
    print("Outcome breakdown:")
    print(
        result_vc.rename('count')
        .to_frame()
        .assign(pct_=(result_vc / result_vc.sum() * 100).round(1))
        .sort_values('count', ascending=False)
        .to_string()
    )
else:
    print("No 'Eindresultaat' tab found in this export.")


## 7. Case Resolution & Timing

Case status breakdown and time from first signal to case closure.


In [ ]:
_report_section('7. Case Resolution & Timing')
meldingen['afgerond_bool']    = ja_nee(meldingen['Afgerond'])
meldingen['ingetrokken_bool'] = ja_nee(meldingen['Ingetrokken'])
meldingen['onterecht_bool']   = ja_nee(meldingen['Onterecht'])
if 'reached_case' not in meldingen.columns:
    meldingen['reached_case'] = False

earliest_signal = (signalen.dropna(subset=['Meldingnummer'])
                   .groupby('Meldingnummer')['Datum melding'].min()
                   .rename('Datum melding'))
mel = meldingen.join(earliest_signal, on='Meldingnummer')
mel['days_to_close'] = (mel['Afgerond d.d.'] - mel['Datum melding']).dt.days

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_mel = len(meldingen)
status = pd.Series({
    'Resolved\n(Afgerond)':     int(meldingen['afgerond_bool'].sum()),
    'Reached\n(via contact)':   int(meldingen['reached_case'].sum()),
    'Withdrawn\n(Ingetrokken)': int(meldingen['ingetrokken_bool'].sum()),
    'Incorrect\n(Onterecht)':   int(meldingen['onterecht_bool'].sum()),
})
bars = axes[0].bar(status.index, status.values, color=COLORS[:4])
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width() / 2, h + 0.3,
                 f'{int(h):,}\n({h/max(n_mel,1)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=9)
axes[0].set_title(f'Report Status Overview  (n={n_mel:,})')
axes[0].set_ylabel('Count')

dtc = mel['days_to_close'].dropna()
dtc = dtc[dtc >= 0]
if len(dtc) > 1:
    dtc.hist(ax=axes[1], bins=min(30, len(dtc)), color=COLORS[2], edgecolor='white')
    axes[1].axvline(dtc.median(), color='red', linestyle='--', linewidth=1.5,
                    label=f'Median: {dtc.median():.0f}d')
    axes[1].axvline(dtc.mean(), color='orange', linestyle=':', linewidth=1.5,
                    label=f'Mean: {dtc.mean():.0f}d')
    axes[1].set_title('Days from Signal to Case Closure')
    axes[1].set_xlabel('Days'); axes[1].set_ylabel('Count'); axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'Insufficient data\nfor time-to-close',
                ha='center', va='center', transform=axes[1].transAxes, fontsize=11)

plt.tight_layout(); plt.show()

print("Status summary (of all cases in the export):")
for label, count in status.items():
    print(f"  {label.replace(chr(10), ' '):<30} {count:>6,}  ({count/max(n_mel,1)*100:.1f}%)")
print("  Note: 'Reached' is case-level from contact attempts; days-to-close covers only")
print("        cases whose opening signal is still in the 6-month window.")
if len(dtc) > 1:
    print(f"\nTime to close (days) - percentiles behind the histogram:")
    print(f"  mean={dtc.mean():.0f}  p10={dtc.quantile(.1):.0f}  p25={dtc.quantile(.25):.0f}  "
          f"p50={dtc.median():.0f}  p75={dtc.quantile(.75):.0f}  p90={dtc.quantile(.9):.0f}")


### 7a. Recidive — Repeated Signals

A recidive signal means the same household has sent a signal before.
This cell shows how the recidive rate evolves month by month and whether households with a recidive history are more or less likely to be forwarded to a report.


In [ ]:
_report_section('7a. Recidive — Repeated Signals')
recidive_cols = [c for c in signalen.columns if c.startswith('Recidive maand')]

if recidive_cols:
    rec_rates = {col.replace('Recidive ', ''): ja_nee(signalen[col]).mean() * 100
                 for col in recidive_cols}
    overall_rec = ja_nee(signalen['Recidive']).mean() * 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    pd.Series(rec_rates).plot.bar(ax=axes[0], color=COLORS[0])
    axes[0].axhline(overall_rec, color='red', linestyle='--', linewidth=1.5,
                    label=f'Overall: {overall_rec:.1f}%')
    axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[0].set_title('Recidive Rate by Month Prior')
    axes[0].set_ylabel('% of signals flagged as recidive')
    axes[0].tick_params(axis='x', rotation=0)
    axes[0].legend()

    rec_conv = (signalen
                .groupby(ja_nee(signalen['Recidive']).rename('recidive'))
                ['doorgezet_bool'].mean() * 100)
    rec_conv.index = rec_conv.index.map({True: 'Recidive', False: 'No Recidive'})
    rec_conv.plot.bar(ax=axes[1], color=[COLORS[1], COLORS[2]])
    axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[1].set_title('Forwarding Rate: Recidive vs No Recidive')
    axes[1].set_ylabel('% Forwarded to Report (Doorgezet)')
    axes[1].tick_params(axis='x', rotation=0)

    plt.tight_layout()
    plt.show()

    print(f"Overall recidive rate: {overall_rec:.1f}%")
    print(pd.Series(rec_rates).round(1).rename('recidive_%').to_string())
    print()
    print("Forwarding rate by recidive status (values behind the right chart):")
    print(rec_conv.round(1).rename('forwarded_%').to_frame().to_string())
else:
    print("No recidive month columns found.")


## 8. End-to-End Effectiveness Funnel

The journey from signal receipt to resolved case. Because signals older than ~6 months are
deleted while cases persist for years, `len(Meldingen)` is **not** a subset of the current
signals — a raw funnel would exceed 100%. So this funnel is restricted to the **current
6-month signal cohort**: signals received → forwarded → their case → reached → wants help →
resolved, all as a percentage of signals received in the window.


In [ ]:
_report_section('8. End-to-End Effectiveness Funnel')
# Restrict to the CURRENT 6-month signal cohort so every stage shares one denominator.
# (Meldingen persist for years but old signals are deleted, so len(meldingen) is NOT a
#  subset of current signals; a raw signal->case funnel would exceed 100%.)
_dz = ja_nee(signalen['Doorgezet']) == True
cohort_mnums = set(signalen.loc[_dz, 'Meldingnummer'].dropna())
cohort_mld = meldingen[meldingen['Meldingnummer'].isin(cohort_mnums)]
if 'reached_case' not in cohort_mld.columns:
    cohort_mld = cohort_mld.assign(reached_case=False)

n_signals   = len(signalen)
n_forwarded = int(_dz.sum())
n_reports   = len(cohort_mld)
n_reached   = int(cohort_mld['reached_case'].sum())
n_resolved  = int(ja_nee(cohort_mld['Afgerond']).sum())

stages = ['Signals received (last 6 mo)',
          'Forwarded to report\n(Doorgezet = Ja)',
          'Report/case exists',
          'Client reached\n(via contact attempts)']
values = [n_signals, n_forwarded, n_reports, n_reached]

if 'Eindresultaat' in extra_tabs:
    e_co = extra_tabs['Eindresultaat']
    e_co = e_co[e_co['Meldingnummer'].isin(cohort_mnums)]
    hulp_ja = int(e_co['Wil de klant hulp'].fillna('').astype(str).str.lower().str.startswith('ja').sum())
    stages.append('Client wants help\n(Wil hulp = Ja)')
    values.append(hulp_ja)

stages.append('Case resolved\n(Afgerond = Ja)')
values.append(n_resolved)

palette = sns.color_palette('Blues_r', len(stages))
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(stages)), values, color=palette)
for bar, val in zip(bars, values):
    pct = val / max(n_signals, 1) * 100
    ax.text(bar.get_width() + n_signals * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:,}   ({pct:.1f}% of signals)', va='center', fontsize=10)
ax.set_yticks(range(len(stages))); ax.set_yticklabels(stages, fontsize=11)
ax.set_xlabel('Count')
ax.set_title('End-to-End Funnel - current 6-month signal cohort', fontsize=14, pad=15)
ax.set_xlim(0, n_signals * 1.45); ax.invert_yaxis()
plt.tight_layout(); plt.show()

print("Funnel (restricted to signals from the last ~6 months; one consistent denominator):")
for stage, val in zip(stages, values):
    print(f"  {stage.replace(chr(10), ' '):<44} {val:>6,}  ({val/max(n_signals,1)*100:.1f}%)")
print("\nNote: older cases whose signals were deleted are excluded on purpose - a raw")
print("signal->case funnel would exceed 100% because cases persist but signals do not.")


## 9. Effectiveness Framework — Does Contact Reduce Return?

Sections 9–12 replace the old forward-window recidive analysis with an **effectiveness** framework
that is complementary to Divosa's work. Divosa models *reach* (was the client contacted?); here
"reached" is an **input**, and the outcome is whether the **household returns with a new signal**.

**Outcome:** time from **case close** (`Afgerond d.d.`) until the same household `Hash` sends a new
signal, **right-censored** at the export date — a survival analysis (Kaplan–Meier + Cox). Censoring
is what makes this work on a short (6-month) window: households not yet returned simply contribute
partial follow-up instead of being discarded.

This first cell prepares the shared analysis frame (`eff`, one row per closed, linked case).
All experiments are single-municipality; results are associational, not causal.

In [ ]:
_report_section('9. Effectiveness Framework — data preparation')
import numpy as np
import bisect
from statsmodels.duration.survfunc import SurvfuncRight, survdiff
from statsmodels.duration.hazard_regression import PHReg

EXPORT_DATE = signalen['Datum melding'].max()
print(f"Export (last signal) date: {EXPORT_DATE.date() if pd.notna(EXPORT_DATE) else 'n/a'}")

# ── Clean age (raw column contains impossible values e.g. -7973, 125) ─────────
signalen['Leeftijd_clean'] = signalen['Leeftijd'].where(signalen['Leeftijd'].between(18, 99))

# ── Signal-level helper flags ─────────────────────────────────────────────────
signalen['doorgezet_bool'] = ja_nee(signalen['Doorgezet'])
signalen['recidive_hist']  = ja_nee(signalen['Recidive'])   # built-in backward-looking flag

# reporter type onto signals
if 'Melder ID' in signalen.columns and 'Type melder' in melders.columns:
    _mel = melders.rename(columns={'ID': 'Melder ID'})[['Melder ID', 'Type melder']]
    signalen_x = signalen.merge(_mel, on='Melder ID', how='left')
else:
    signalen_x = signalen.copy(); signalen_x['Type melder'] = np.nan

# debt & age bands (ascii labels — safe for regression formulas)
_debt_col = next((c for c in ['Achterstandsbedrag', 'Termijnbedrag'] if c in signalen_x.columns), None)
if _debt_col:
    signalen_x['debt_band'] = pd.cut(signalen_x[_debt_col].clip(lower=0),
        bins=[-1, 200, 700, 2000, 1e12], labels=['<200', '200-700', '700-2k', '>2k'])
signalen_x['age_band'] = pd.cut(signalen_x['Leeftijd_clean'],
    bins=[18, 30, 45, 60, 100], right=False, labels=['18-29', '30-44', '45-59', '60+'])

# ── Case -> originating (earliest) signal + its covariates ────────────────────
_slink = signalen_x.dropna(subset=['Meldingnummer']).sort_values('Datum melding')
_first = (_slink.groupby('Meldingnummer', as_index=False).first()
          .rename(columns={'Datum melding': 'signal_date'}))

# ── Contact tabs -> case-level reach / attempts / first-reach date ─────────────
_cparts = []
for _t in ['Tussenresultaat', 'Eindresultaat']:
    if _t in extra_tabs:
        _d = extra_tabs[_t].copy(); _d['_tab'] = _t
        _cparts.append(_d)
contacts = pd.concat(_cparts, ignore_index=True) if _cparts else pd.DataFrame()

if len(contacts):
    contacts['_reached'] = ja_nee(contacts['Persoon bereikt'])
    _g = contacts.groupby('Meldingnummer')
    case_contact = pd.DataFrame({'n_attempts': _g.size(),
                                 'reached': _g['_reached'].any()})
    _succ = contacts[contacts['_reached'] == True]
    case_contact = case_contact.join(
        _succ.groupby('Meldingnummer')['Datum contactpoging'].min().rename('first_reach_date'))
    case_contact = case_contact.reset_index()
    _m = contacts.dropna(subset=['Wijze van contactpoging'])
    case_method = (_m.groupby('Meldingnummer')['Wijze van contactpoging']
                   .agg(lambda s: s.value_counts().idxmax()).rename('primary_method').reset_index())
else:
    case_contact = pd.DataFrame(columns=['Meldingnummer', 'n_attempts', 'reached', 'first_reach_date'])
    case_method  = pd.DataFrame(columns=['Meldingnummer', 'primary_method'])

# help decision + final result: take the last non-blank from Eindresultaat (final word)
def _last_nonblank(s):
    s = s.dropna()
    return s.iloc[-1] if len(s) else np.nan
eind = extra_tabs.get('Eindresultaat', pd.DataFrame())
if len(eind):
    _e = eind.sort_values('Datum contactpoging').groupby('Meldingnummer')
    case_eind = pd.DataFrame({'wil_hulp': _e['Wil de klant hulp'].agg(_last_nonblank),
                              'eindresultaat': _e['Wat is het eindresultaat'].agg(_last_nonblank)}).reset_index()
else:
    case_eind = pd.DataFrame(columns=['Meldingnummer', 'wil_hulp', 'eindresultaat'])

# ── Closed cases -> master survival frame ─────────────────────────────────────
_mc = meldingen[['Meldingnummer', 'Afgerond d.d.']].copy()
_mc['afgerond'] = ja_nee(meldingen['Afgerond'])
_mc = _mc[(_mc['afgerond'] == True) & _mc['Afgerond d.d.'].notna()].rename(columns={'Afgerond d.d.': 'close_date'})

eff = _mc.merge(_first[['Meldingnummer', 'Hash', 'signal_date', 'doorgezet_bool',
                        'recidive_hist', 'Type melder', 'debt_band', 'age_band', 'Leeftijd_clean']],
                on='Meldingnummer', how='inner')

for _bc in ['debt_band', 'age_band']:
    if _bc in eff.columns and hasattr(eff[_bc], 'cat'):
        eff[_bc] = eff[_bc].cat.remove_unused_categories()

# next signal from the same Hash strictly after close_date (searchsorted per household)
_allsig = signalen.dropna(subset=['Hash', 'Datum melding'])
_sig_by_hash = _allsig.sort_values('Datum melding').groupby('Hash')['Datum melding'].apply(list).to_dict()
def _next_sig(h, c):
    arr = _sig_by_hash.get(h)
    if not arr:
        return pd.NaT
    i = bisect.bisect_right(arr, c)
    return arr[i] if i < len(arr) else pd.NaT
eff['next_signal_date'] = [_next_sig(h, c) for h, c in zip(eff['Hash'], eff['close_date'])]

for _dc in ['close_date', 'signal_date', 'next_signal_date']:
    if _dc in eff.columns:
        eff[_dc] = pd.to_datetime(eff[_dc], errors='coerce')

eff['event'] = eff['next_signal_date'].notna().astype(int)
_end = eff['next_signal_date'].fillna(EXPORT_DATE)
eff['duration'] = (_end - eff['close_date']).dt.days
eff = eff[eff['duration'] > 0].copy()               # closed before export; positive follow-up

# ── Signal-level survival frame (for A4: non-forwarded signals never become cases) ──
_ss = signalen_x.dropna(subset=['Hash', 'Datum melding']).copy()
_ss['signal_date'] = pd.to_datetime(_ss['Datum melding'], errors='coerce')
_ss = _ss.dropna(subset=['signal_date'])
def _next_after(h, t):
    arr = _sig_by_hash.get(h)
    if not arr:
        return pd.NaT
    j = bisect.bisect_right(arr, t)
    return arr[j] if j < len(arr) else pd.NaT
_ss['next_date'] = [_next_after(h, t) for h, t in zip(_ss['Hash'], _ss['signal_date'])]
_ss['event'] = _ss['next_date'].notna().astype(int)
_ss['duration'] = (_ss['next_date'].fillna(EXPORT_DATE) - _ss['signal_date']).dt.days
sigsurv = _ss[_ss['duration'] > 0].copy()
for _bc in ['debt_band', 'age_band']:
    if _bc in sigsurv.columns and hasattr(sigsurv[_bc], 'cat'):
        sigsurv[_bc] = sigsurv[_bc].cat.remove_unused_categories()

# attach contact-derived case features
for _cf in [case_contact, case_eind, case_method]:
    if len(_cf):
        eff = eff.merge(_cf, on='Meldingnummer', how='left')
if 'reached' not in eff.columns:    eff['reached'] = False
if 'n_attempts' not in eff.columns: eff['n_attempts'] = np.nan
eff['reached'] = eff['reached'].fillna(False)
if 'first_reach_date' in eff.columns:
    eff['first_reach_date'] = pd.to_datetime(eff['first_reach_date'], errors='coerce')
    eff['days_to_reach'] = (eff['first_reach_date'] - eff['signal_date']).dt.days
else:
    eff['days_to_reach'] = np.nan

# ── Arm assignment (short codes -> labels) ────────────────────────────────────
ARM_LABEL = {'A_not_reached': 'A - not reached', 'B_refused': 'B - help refused',
             'C_accepted': 'C - help accepted',
             'B1_resolving': 'B1 - declined (already resolving)',
             'B2_difficulty': 'B2 - declined (still in difficulty)'}
_ACCEPT = {'ja', 'vervolg afspraak gemaakt', 'wenst contact met vst'}
def _arm(row):
    if not bool(row['reached']):
        return 'A_not_reached'
    hulp = str(row.get('wil_hulp', '') or '').strip().lower()
    return 'C_accepted' if hulp in _ACCEPT else 'B_refused'
def _arm_clean(row):
    a = row['arm']
    if a != 'B_refused':
        return a
    eind = str(row.get('eindresultaat', '') or '').lower()
    if any(k in eind for k in ['betaald', 'betalingsregeling', 'zelf probeert', 'quick fix', 'schuldhulp']):
        return 'B1_resolving'
    return 'B2_difficulty'
if len(eff):
    eff['arm'] = eff.apply(_arm, axis=1)
    eff['arm_clean'] = eff.apply(_arm_clean, axis=1)
else:
    eff['arm'] = pd.Series(dtype=object)
    eff['arm_clean'] = pd.Series(dtype=object)

# ── Shared survival helpers ───────────────────────────────────────────────────
def cuminc_at(dur, evt, day):
    dur = np.asarray(dur, float); evt = np.asarray(evt, int)
    if len(dur) < 5 or evt.sum() == 0:
        return np.nan
    sf = SurvfuncRight(dur, evt)
    idx = np.searchsorted(sf.surv_times, day, side='right') - 1
    if idx < 0:
        return 0.0
    return float(1 - sf.surv_prob[idx])

def cox_hr(data, formula, covars, event_col='event', dur_col='duration'):
    d = data.dropna(subset=[dur_col, event_col] + covars).copy()
    if len(d) < 40 or int(d[event_col].sum()) < 5:
        print('  (Cox skipped - too few cases/events)'); return None
    try:
        r = PHReg.from_formula(formula, status=d[event_col].values, data=d).fit()
        _names = list(r.model.exog_names)
        _params = np.asarray(r.params); _pvals = np.asarray(r.pvalues)
        _ci = np.asarray(r.conf_int())
        return pd.DataFrame({'HR': np.exp(_params), 'CI2.5%': np.exp(_ci[:, 0]),
                             'CI97.5%': np.exp(_ci[:, 1]), 'p': _pvals},
                            index=_names).round(3)
    except Exception as _e:
        print(f'  Cox failed: {_e}'); return None

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Closed, linked cases in effectiveness frame: {len(eff):,}")
if len(eff):
    print(f"  returned (event):        {int(eff['event'].sum()):,} ({eff['event'].mean()*100:.1f}%)")
    print(f"  censored at export:      {int((eff['event']==0).sum()):,}")
    print(f"  median follow-up (days): {eff['duration'].median():.0f}")
    print(f"  reached (any contact):   {int(eff['reached'].sum()):,} ({eff['reached'].mean()*100:.1f}%)")
    print("  arm sizes:")
    for _c, _n in eff['arm'].value_counts().items():
        print(f"    {ARM_LABEL.get(_c, _c):<34} {_n:,}")
else:
    print("  No closed+linked cases — check the export (needs Meldingnummer on signals + close dates).")

### 9a. Contact Decomposition — the "wake-up call" test (A1 + A2)

Splits closed cases into three arms and compares time-to-return:
- **A** not reached, **B** reached but help refused, **C** reached and help accepted.

The key contrast is **B vs A**: do households that were reached but *declined* help still return
less than those never reached? That would indicate contact works partly through *awareness*, not
only through help. **A2** guards against reverse causation by splitting B into *B1* (declined because
already resolving/paid — excluded) and *B2* (declined while still in difficulty — the clean test).

In [ ]:
_report_section('9. Contact Decomposition - Does Contact Reduce Return?')

_arms = ['A_not_reached', 'B_refused', 'C_accepted']
_present = [a for a in _arms if (eff['arm'] == a).sum() >= 20]

if len(eff) == 0 or int(eff['event'].sum()) < 10 or len(_present) < 2:
    print('Not enough events / arms to compare (needs longer follow-up or more cases).')
else:
    fig, ax = plt.subplots(figsize=(11, 6))
    for a in _present:
        g = eff[eff['arm'] == a]
        sf = SurvfuncRight(g['duration'].values, g['event'].values)
        ax.step(sf.surv_times, (1 - sf.surv_prob) * 100, where='post',
                linewidth=2, label=f'{ARM_LABEL[a]} (n={len(g)})')
    ax.set_xlim(0, 270)
    ax.set_xlabel('Days since case close'); ax.set_ylabel('Cumulative return %')
    ax.set_title('Return after case close, by contact arm')
    ax.legend()
    plt.tight_layout(); plt.show()

    _HZ = [30, 60, 90, 120, 150, 180, 210, 240]
    print('Cumulative return % by arm (values behind the curves):')
    print('  ' + 'arm'.ljust(20) + ''.join(f'{h}d'.rjust(8) for h in _HZ) + '       n')
    for a in _present:
        g = eff[eff['arm'] == a]
        print('  ' + ARM_LABEL[a].ljust(20)
              + ''.join(f'{cuminc_at(g["duration"], g["event"], h)*100:7.1f}%' for h in _HZ)
              + f'{len(g):8d}')

    try:
        _sub = eff[eff['arm'].isin(_present)]
        _chi, _p = survdiff(_sub['duration'].values, _sub['event'].values, _sub['arm'].values)
        print(f'\nLog-rank across arms: chi2={_chi:.2f}, p={_p:.4f}')
    except Exception as _e:
        print(f'Log-rank failed: {_e}')

    print('\nAdjusted hazard ratios (ref = A not reached; HR<1 = fewer/slower returns):')
    hr = cox_hr(eff, 'duration ~ C(arm, Treatment(reference="A_not_reached")) + recidive_hist '
                     '+ C(debt_band) + C(age_band)',
                ['arm', 'recidive_hist', 'debt_band', 'age_band'])
    if hr is not None:
        display(hr)

    # ── A2: clean contrast — refusers still in difficulty (B2) vs not reached (A) ──
    print('\nA2 - Clean test (excludes B1 already-resolving refusers):')
    for a in ['A_not_reached', 'B2_difficulty', 'C_accepted']:
        g = eff[eff['arm_clean'] == a]
        if len(g) >= 20:
            print(f'  {ARM_LABEL[a]:<34} 90d return={cuminc_at(g["duration"], g["event"], 90)*100:5.1f}%  (n={len(g)})')

## 10. Reach vs Effectiveness, and Resolution Quality (A3 + A4 + A5)

- **A3 (the standout):** Divosa ranks contact methods by *reach* (phone > home-visit > email > sms).
  Does the method that best *reaches* also best *prevent return*? If the orderings diverge,
  optimising for reach is not the same as optimising for effectiveness.
- **A4:** does forwarding a signal to a full case (`Doorgezet = Ja`) reduce later return?
- **A5:** does a *good* final resolution (paid / arrangement) predict not returning?

In [ ]:
_report_section('10. Reach vs Effectiveness & Resolution Quality')

# ── A4 - forwarding -> next signal (signal-level; non-forwarded signals never become cases) ──
print('A4 - Forwarding (Doorgezet) -> next signal  [signal-level Cox HR, ref = not forwarded]:')
hr = cox_hr(sigsurv, 'duration ~ doorgezet_bool + recidive_hist', ['doorgezet_bool', 'recidive_hist'])
if hr is not None:
    display(hr)

# ── A3 - does the best-reaching method also best prevent return? ──────────────
print('\nA3 - 90-day return by primary contact method (reached cases, n>=30):')
if 'primary_method' in eff.columns:
    _m = eff[eff['reached'] == True].dropna(subset=['primary_method'])
    _rows = []
    for meth, g in _m.groupby('primary_method'):
        if len(g) >= 30:
            _rows.append((str(meth)[:45], len(g), cuminc_at(g['duration'], g['event'], 90) * 100))
    if _rows:
        _mt = pd.DataFrame(_rows, columns=['method', 'n', 'return_90d_%']).sort_values('return_90d_%')
        display(_mt.round(1))
        fig, ax = plt.subplots(figsize=(11, 5))
        ax.barh(_mt['method'], _mt['return_90d_%'], color=COLORS[0])
        ax.set_xlabel('90-day return %  (lower = more effective at preventing return)')
        ax.set_title('A3 - Return by contact method')
        plt.tight_layout(); plt.show()
        print('Compare this ordering with Divosa reach ranking (phone > home-visit > email > sms):')
        print('divergence => reach-effective method is NOT the same as return-preventing method.')
    else:
        print('  No method with >=30 reached cases.')
else:
    print('  primary_method unavailable.')

# ── A5 - does a good resolution predict not returning? ────────────────────────
print('\nA5 - 90-day return by final result (Eindresultaat, n>=30):')
if 'eindresultaat' in eff.columns:
    _rows = []
    for res, g in eff.dropna(subset=['eindresultaat']).groupby('eindresultaat'):
        if len(g) >= 30:
            _rows.append((str(res)[:55], len(g), cuminc_at(g['duration'], g['event'], 90) * 100))
    if _rows:
        display(pd.DataFrame(_rows, columns=['eindresultaat', 'n', 'return_90d_%'])
                .sort_values('return_90d_%').round(1))
    else:
        print('  No result category with >=30 cases.')
else:
    print('  eindresultaat unavailable.')

## 11. Targeting — Who Returns? (B1 + B2 + B3)

Descriptive targeting using the built-in `Recidive` flag (computed by the source system against its
full history, so it is trustworthy and fully observable now). Profiles the repeat-signal rate across
household dimensions to answer *who to prioritise* — complementing Divosa's *who is easiest to reach*.
Groups with fewer than 30 signals are excluded.

In [ ]:
_report_section('11. Targeting - Who Returns? (built-in Recidive flag)')
MIN_CELL = 30
obsB = signalen_x.copy()
obsB['recidive_hist'] = ja_nee(obsB['Recidive'])
_ov = obsB['recidive_hist'].mean() * 100
print(f"Overall built-in recidive rate: {_ov:.1f}%  (n={int(obsB['recidive_hist'].notna().sum()):,})")

_dims = [('Type melder', 'Reporter type'), ('debt_band', 'Debt band'),
         ('age_band', 'Age band'), ('Minderjarige kinderen', 'Minor children')]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, (col, label) in zip(axes.flatten(), _dims):
    if col not in obsB.columns:
        ax.text(0.5, 0.5, f'{label}\nnot available', ha='center', va='center', transform=ax.transAxes); continue
    g = obsB.dropna(subset=[col]).groupby(col)['recidive_hist'].agg(n='count', rate='mean')
    g = g[g['n'] >= MIN_CELL].sort_values('rate')
    if len(g) == 0:
        ax.text(0.5, 0.5, f'{label}\ntoo few', ha='center', va='center', transform=ax.transAxes); continue
    ax.barh(g.index.astype(str), g['rate'] * 100, color=COLORS[1])
    ax.axvline(_ov, color='red', linestyle='--', linewidth=1.5, label=f'overall {_ov:.1f}%')
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title(f'Recidive by {label}'); ax.set_xlabel('Repeat-signal %'); ax.legend(fontsize=8)
plt.suptitle('Targeting - repeat-signal rate by household profile', y=1.01)
plt.tight_layout(); plt.show()

print('\nHighest-risk segment per dimension:')
for col, label in _dims:
    if col not in obsB.columns:
        continue
    g = obsB.dropna(subset=[col]).groupby(col)['recidive_hist'].agg(n='count', rate='mean')
    g = g[g['n'] >= MIN_CELL].sort_values('rate', ascending=False)
    if len(g):
        print(f'  {label:<16}: {g.index[0]}  ->  {g.iloc[0]["rate"]*100:.1f}%  (n={int(g.iloc[0]["n"]):,})')

print('\nFull recidive breakdown by dimension (values behind the bars, n>=30):')
for col, label in _dims:
    if col not in obsB.columns:
        continue
    g = obsB.dropna(subset=[col]).groupby(col)['recidive_hist'].agg(n='count', rate='mean')
    g = g[g['n'] >= MIN_CELL].sort_values('rate', ascending=False)
    if len(g):
        g['recidive_%'] = (g['rate'] * 100).round(1)
        print(f'\\n- {label}:')
        print(g.drop(columns='rate').to_string())

## 12. Timing & Dose-Response (C1–C4)

- **C3** — *when* do households return (overall cumulative-incidence curve)?
- **C1** — does reaching a household *faster* reduce return?
- **C2** — does the *number* of attempts relate to return (dose-response)?
- **C4** — how many signals do households send (chronic vs one-off returners)?

In [ ]:
_report_section('12. Timing & Dose-Response')

# ── C3 - overall recurrence timing ────────────────────────────────────────────
if len(eff) and int(eff['event'].sum()) >= 10:
    sf = SurvfuncRight(eff['duration'].values, eff['event'].values)
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.step(sf.surv_times, (1 - sf.surv_prob) * 100, where='post', color=COLORS[2], linewidth=2)
    ax.set_xlim(0, 270); ax.set_xlabel('Days since case close'); ax.set_ylabel('Cumulative return %')
    ax.set_title('C3 - When do households return? (overall)')
    plt.tight_layout(); plt.show()
    _HZ = [30, 60, 90, 120, 150, 180, 210, 240]
    print("Overall cumulative return % (values behind the curve):")
    print('  ' + ''.join(f'{h}d'.rjust(8) for h in _HZ))
    print('  ' + ''.join(f'{cuminc_at(eff["duration"], eff["event"], h)*100:7.1f}%' for h in _HZ))
else:
    print('C3 - Not enough events for a recurrence curve.')

# ── C1 - speed to contact ─────────────────────────────────────────────────────
print('\nC1 - Speed-to-contact -> return:')
if 'days_to_reach' in eff.columns and eff['days_to_reach'].notna().any():
    _r = eff[(eff['reached'] == True) & eff['days_to_reach'].notna()].copy()
    if len(_r) >= 40:
        _med = _r['days_to_reach'].median()
        _r['speed_grp'] = np.where(_r['days_to_reach'] <= _med, 'fast', 'slow')
        for grp, g in _r.groupby('speed_grp'):
            print(f'  {grp:<5} (n={len(g)}) 90d return={cuminc_at(g["duration"], g["event"], 90)*100:.1f}%'
                  f'  (median days-to-reach split at {_med:.0f})')
        hr = cox_hr(_r, 'duration ~ days_to_reach + recidive_hist', ['days_to_reach', 'recidive_hist'])
        if hr is not None:
            display(hr)
    else:
        print('  Too few reached cases with contact timing.')
else:
    print('  days_to_reach unavailable.')

# ── C2 - contact intensity (dose-response) ────────────────────────────────────
print('\nC2 - Number of attempts -> return:')
if 'n_attempts' in eff.columns and eff['n_attempts'].notna().any():
    _a = eff.dropna(subset=['n_attempts']).copy()
    _a['att_band'] = pd.cut(_a['n_attempts'], bins=[0, 1, 3, 6, 1e9], labels=['1', '2-3', '4-6', '7+'])
    _rows = []
    for b, g in _a.groupby('att_band'):
        if len(g) >= 30:
            _rows.append((str(b), len(g), cuminc_at(g['duration'], g['event'], 90) * 100))
    if _rows:
        display(pd.DataFrame(_rows, columns=['attempts', 'n', 'return_90d_%']).round(1))
    else:
        print('  Too few cases per attempt band.')
else:
    print('  n_attempts unavailable.')

# ── C4 - recurrent signals per household ──────────────────────────────────────
print('\nC4 - Repeated signals per household:')
_sph = signalen.dropna(subset=['Hash']).groupby('Hash').size()
if len(_sph):
    print(f'  households={len(_sph):,}  mean={_sph.mean():.2f}  median={_sph.median():.0f}  '
          f'max={_sph.max()}  with >1 signal: {(_sph > 1).mean()*100:.1f}%')

## 13. Generate the Results File to Send Back

Run this final cell **after running all the cells above**. It writes a single self-contained file —
**`elsa_results_report.html`** — into the same folder as the data, containing every table, chart and
printed summary produced above (charts are embedded directly in the file).

**This is the only file you need to send back.** It opens in any web browser, and to convert it to PDF you
can simply open it and use the browser's *Print → Save as PDF*.

> It contains **aggregated statistics, tables and charts only** — no individual records.


In [ ]:
_report_section('13. Generate the Results File to Send Back')
_doc = f'''<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>ELSA Vroegsignalering — Results</title>
<style>
 body{{font-family:-apple-system,Segoe UI,Arial,sans-serif;max-width:1000px;margin:24px auto;
       padding:0 18px;color:#222;line-height:1.4}}
 h1{{border-bottom:3px solid #2c7fb8;padding-bottom:6px}}
 h2.sec{{background:#2c7fb8;color:#fff;padding:6px 12px;border-radius:4px;margin-top:34px}}
 pre{{background:#f6f8fa;padding:8px 10px;border-radius:4px;white-space:pre-wrap;
      font-size:13px;font-family:SFMono-Regular,Consolas,monospace}}
 table{{border-collapse:collapse;margin:10px 0;font-size:13px}}
 th,td{{border:1px solid #ccc;padding:4px 9px;text-align:right}}
 th{{background:#eef3f7}}
 img{{max-width:100%;margin:12px 0;border:1px solid #eee;border-radius:4px}}
 .meta{{color:#666;font-size:13px}}
</style></head><body>
<h1>ELSA Vroegsignalering — Effectiveness Results</h1>
<p class="meta">Generated {_dt.datetime.now():%Y-%m-%d %H:%M} &middot; source file: {_html.escape(str(FILE))}
&middot; {len(_REPORT_BLOCKS)} content blocks</p>
<p class="meta"><em>Aggregated statistics, tables and charts only — no individual records.</em></p>
{''.join(_REPORT_BLOCKS)}
</body></html>'''

# Write next to the notebook's working directory, timestamped so runs never overwrite
REPORT_PATH = Path(REPORT_PATH).resolve()
REPORT_PATH = REPORT_PATH.with_name(f"{REPORT_PATH.stem}_{_dt.datetime.now():%Y%m%d-%H%M%S}{REPORT_PATH.suffix}")
REPORT_PATH.write_text(_doc, encoding='utf-8')

_orig_print("=" * 70)
_orig_print("RESULTS FILE CREATED — this is the file to send back:")
_orig_print(f"   {REPORT_PATH}")
_orig_print("=" * 70)
_orig_print(f"  {len(_REPORT_BLOCKS)} content blocks captured (text, tables, charts).")
_orig_print(f"  Working directory: {Path.cwd()}")
_orig_print("  Open it in any browser; use Print -> Save as PDF if a PDF is preferred.")

# Clickable download link in Jupyter / JupyterLab
try:
    from IPython.display import FileLink
    _ipy_display(FileLink(str(REPORT_PATH)))
except Exception:
    pass

# In Google Colab, trigger a direct browser download automatically
try:
    from google.colab import files as _colab_files   # noqa
    _colab_files.download(str(REPORT_PATH))
except Exception:
    pass
